In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import gc
import torch
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from PIL import UnidentifiedImageError
import timm  # Transformer models


In [3]:
!pip install grad-cam


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 42.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 41.1 MB/s 

In [4]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt


In [5]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [6]:
# Dataset class
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.label_map = {"normal": 0, "bacterial": 1, "viral": 2}

        for label in self.label_map:
            label_dir = os.path.join(root_dir, label)
            for file in os.listdir(label_dir):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(label_dir, file), self.label_map[label]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):  # <- This must be inside the class, not outside
        path, label = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except UnidentifiedImageError:
            print(f"Skipping corrupted file: {path}")
            return self.__getitem__((idx + 1) % len(self.samples))  # Skip corrupted image

        if self.transform:
            image = self.transform(image)
        return image, label


In [7]:
# Transforms
img_size = 224
transform_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val_test = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])



In [8]:
# Datasets and loaders
train_dataset = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/train", transform_train)
val_dataset   = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/val", transform_val_test)
test_dataset  = ChestXRayDataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images/test", transform_val_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, num_workers=2)


In [9]:
# Corrupted or not
def clean_dataset(folder):
    bad_files = []
    for root, _, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                try:
                    img = Image.open(full_path)
                    img.verify()
                except Exception as e:
                    print("Corrupted:", full_path)
                    bad_files.append(full_path)
    print(f"\nTotal corrupted: {len(bad_files)}")
    for f in bad_files:
        os.remove(f)

# Run this once
clean_dataset("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray_images")


Total corrupted: 0


In [10]:
def count_dataset_classes(dataset, dataset_name):
    label_map_inv = {0: 'normal', 1: 'bacterial', 2: 'viral'}
    label_counts = Counter(label for _, label in dataset.samples)

    print(f"\n-->  {dataset_name.upper()} DATASET:")
    for label, count in sorted(label_counts.items()):
        class_name = label_map_inv.get(label, f'class_{label}')
        print(f"  {class_name}: {count} images")

# Count for each dataset
count_dataset_classes(train_dataset, "train")
count_dataset_classes(val_dataset, "val")
count_dataset_classes(test_dataset, "test")


-->  TRAIN DATASET:
  normal: 1808 images
  bacterial: 1945 images
  viral: 1827 images

-->  VAL DATASET:
  normal: 259 images
  bacterial: 278 images
  viral: 261 images

-->  TEST DATASET:
  normal: 517 images
  bacterial: 556 images
  viral: 523 images


In [11]:

# ResFormer (CNN + ViT Hybrid Model)
class ResFormer(nn.Module):
    def __init__(self, num_classes=3):
        super(ResFormer, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        self.cnn_out_dim = 2048

        # Use ViT Base (768 features)
        self.transformer = timm.create_model('vit_base_patch16_224', pretrained=True)
        self.transformer.head = nn.Identity()

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(2048 + 768, num_classes)  # 2816

    def forward(self, x):
        cnn_feat = self.cnn(x)
        cnn_feat_pooled = self.avgpool(cnn_feat).squeeze(-1).squeeze(-1)

        vit_feat = self.transformer(x)
        combined = torch.cat((cnn_feat_pooled, vit_feat), dim=1)

        # Print once when batch size == 32 (or any batch size you expect)
        if combined.shape[0] == 32 and not hasattr(self, 'printed_shape'):
            print("CNN feature shape:", cnn_feat_pooled.shape)
            print("ViT feature shape:", vit_feat.shape)
            print("Combined feature shape:", combined.shape)
            self.printed_shape = True  # Prevent future prints

        return self.fc(combined)




In [12]:
save_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/Models/1.ResFormer.pth"

def train_model(model, save_path, epochs=10):
    model.train()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)

    best_val_acc = 0.0

    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        val_acc = validate_model(model)
        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {100*correct/total:.2f}%, Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

            if os.path.exists(save_path):
                print(f"Saved new best model at epoch {epoch+1} to: {save_path}")
            else:
                print("Model save failed.")

In [13]:
#  Validation function
def validate_model(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100 * correct / total


In [14]:
#  Evaluation function
def evaluate_model(model):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

            if idx % 10 == 0:
                print(f"Processed {idx * len(images)} samples...")

    print("\nConfusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["normal", "bacterial", "viral"]))



In [15]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
model = ResFormer().to(device)
train_model(model, save_path)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 95.6MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [ ]:

model.load_state_dict(torch.load("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/Models/1.ResFormer.pth"))  # ← This loads the best model
model.eval()
evaluate_model(model)

In [ ]:
print("\nTesting Accuracy:")
model.eval()
evaluate_model(model)

In [ ]:
import matplotlib.pyplot as plt

train_acc_list = []
val_acc_list = []
loss_list = []

def train_model(model, epochs=10):
    ...
    for epoch in range(epochs):
        ...
        epoch_acc = 100 * correct / total
        val_acc = validate_model(model)

        train_acc_list.append(epoch_acc)
        val_acc_list.append(val_acc)
        loss_list.append(total_loss / len(train_loader))

        print(f"Epoch {epoch+1} - Train Acc: {epoch_acc:.2f}%, Val Acc: {val_acc:.2f}%")

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(train_acc_list, label="Train Accuracy")
plt.plot(val_acc_list, label="Val Accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(loss_list, label="Train Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

In [ ]:
target_layer = model.cnn[-1]  # Last CNN layer of ResNet
cam = GradCAMPlusPlus(model=model, target_layers=[target_layer], use_cuda=True)

# Pick a test image
image, label = test_dataset[10]
input_tensor = image.unsqueeze(0).to(device)

# Create CAM
targets = [ClassifierOutputTarget(label)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

# Convert input to numpy image
image_np = image.permute(1, 2, 0).numpy()
image_np = image_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
image_np = np.clip(image_np, 0, 1)

# Show overlay
cam_image = show_cam_on_image(image_np, grayscale_cam, use_rgb=True)
plt.imshow(cam_image)
plt.title(f"Grad-CAM++ for Class: {label}")
plt.axis('off')
plt.show()